# PDF Processing Pipeline: Complete Workflow

This notebook processes scientific PDFs through table reconstruction, masking, text extraction, and database ingestion.

## Pipeline Overview:
1. **Setup** - Imports and configuration
2. **Original PDF Processing** - Layout extraction and table reconstruction
3. **PDF Masking** - Remove tables/figures and re-extract layout
4. **Text Processing** - Group by hierarchical path and stitch paragraphs
5. **Media Extraction** - Crop and save tables/figures
6. **Database Ingestion** - Save to PostgreSQL with hierarchical structure

**Date**: 2025-12-31

## 1. Setup

### 1.1 Imports and Module Loading

In [94]:
import sys
import json
from pathlib import Path
import fitz
from IPython.display import Image, display
import pandas as pd
import importlib.util
from collections import Counter
from datetime import datetime

project_root = Path.cwd()
sys.path.insert(0, str(project_root))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

mask_tables = load_module('mask_tables', project_root / 'scripts/docling_files/mask_tables.py')
visualize = load_module('visualize', project_root / 'scripts/visualize_docling_full.py')
text_proc = load_module('text_proc', project_root / 'parsers/text_processing.py')

process_pdf_with_masking = mask_tables.process_pdf_with_masking
reconstruct_tables_from_lists = visualize.reconstruct_tables_from_lists
visualize_pdfs = visualize.visualize_full_layout
ContextAwareStitcher = text_proc.ContextAwareStitcher
remove_citations = text_proc.remove_citations

print("✅ Imports successful!")

✅ Imports successful!


### 1.2 Configure Paths and Directories

In [95]:
PDF_PATH = Path('files/organized_pdfs/PMC1448691_his_2369.pdf')
PMCID = 'PMC1448691'

DOCLING_OUTPUT_DIR = Path('out/docling_full')
MASKED_PDF_DIR = Path('out/masked_pdfs')
TEXT_OUTPUT_DIR = Path('out/text')
TABLES_DIR = Path('files/tables')
FIGURES_DIR = Path('files/figures')
VISUALIZATION_DIR = Path('out/visualization')

for d in [
    DOCLING_OUTPUT_DIR, MASKED_PDF_DIR, TEXT_OUTPUT_DIR, 
    TABLES_DIR, FIGURES_DIR, VISUALIZATION_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📄 PDF: {PDF_PATH.name}")
print(f"📁 PMCID: {PMCID}")

📄 PDF: PMC1448691_his_2369.pdf
📁 PMCID: PMC1448691


## 2. Original PDF Processing

### 2.1 Extract Layout with Docling

Extract document structure from the original PDF including text, tables, figures, and bounding boxes.

In [96]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
import re

CAPTION_PATTERN = re.compile(r'^(Table|Figure)\s+\d+', re.IGNORECASE)

pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = False
pipeline_options.do_ocr = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

print(f"🔄 Extracting layout from ORIGINAL PDF...")
result = converter.convert(str(PDF_PATH))
doc = result.document

print(doc)

all_elements = []
for element, level in doc.iterate_items():
    label = str(getattr(element, "label", "UNKNOWN")).split('.')[-1].upper()
    if not (hasattr(element, 'prov') and element.prov):
        continue
    prov = element.prov[0]
    bbox = prov.bbox
    text = ""
    if hasattr(element, 'text'):
        text = element.text
    elif hasattr(element, 'caption') and element.caption:
        text = element.caption.text
    all_elements.append({
        "type": label,
        "page": prov.page_no,
        "level": level,
        "bbox": {"x1": bbox.l, "y1": bbox.t, "x2": bbox.r, "y2": bbox.b},
        "text": text.strip() if text else None
    })

# Reclassify TEXT elements that match caption patterns (e.g. "Table 6.", "Figure 3.")
reclassified = 0
for el in all_elements:
    if el.get('type') == 'TEXT' and CAPTION_PATTERN.match(el.get('text') or ''):
        el['type'] = 'CAPTION'
        reclassified += 1
if reclassified:
    print(f"🔄 Reclassified {reclassified} TEXT elements as CAPTION")

docling_json_path = DOCLING_OUTPUT_DIR / f"{PDF_PATH.stem}_full_layout.json"

with open(docling_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(PDF_PATH), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc.pages.items()},
        "elements": all_elements
    }, f, indent=2)

types = Counter([el['type'] for el in all_elements])
print(f"\n✅ Original PDF: {len(all_elements)} elements")
print(f"   Tables: {types.get('TABLE', 0)}")
print(f"   Figures: {types.get('PICTURE', 0)}")
print(f"   Captions: {types.get('CAPTION', 0)}")
print(f"💾 {docling_json_path}")

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Initializing pipeline for StandardPdfPipeline with options hash 06b2cfbc41861ca6858eec36b7d53267
INFO: Auto OCR model selected ocrmac.
INFO: Accelerator device: 'mps'


🔄 Extracting layout from ORIGINAL PDF...


INFO: Processing document PMC1448691_his_2369.pdf
INFO: Finished converting document PMC1448691_his_2369.pdf in 31.31 sec.


schema_name='DoclingDocument' version='1.8.0' name='PMC1448691_his_2369' origin=DocumentOrigin(mimetype='application/pdf', binary_hash=14375195741948125159, filename='PMC1448691_his_2369.pdf', uri=None) furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) body=GroupItem(self_ref='#/body', parent=None, children=[RefItem(cref='#/texts/0'), RefItem(cref='#/texts/1'), RefItem(cref='#/texts/2'), RefItem(cref='#/texts/3'), RefItem(cref='#/texts/4'), RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/texts/9'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/texts/12'), RefItem(cref='#/texts/13'), RefItem(cref='#/texts/14'), RefItem(cref='#/texts/15'), RefItem(cref='#/texts/16'), RefItem(cref='#/texts/17'), RefItem(cref='#/texts/18'), RefItem(cref='#/texts/

### 2.2 Reconstruct Tables from Captions

Group table captions with their content rows to create unified table elements.

In [ ]:
print("🔄 Reconstructing tables...")
reconstructed_elements = reconstruct_tables_from_lists(str(docling_json_path))
reconstructed_tables = [el for el in reconstructed_elements if el.get('type') == 'RECONSTRUCTED_TABLE']
print(f"✅ Created {len(reconstructed_tables)} reconstructed tables")

# Keep a snapshot before overwriting so the direct|no-reconstruct combo is available later.
all_elements_raw = list(all_elements)

# Replace all_elements with the merged result: sub-elements that form
# a RECONSTRUCTED_TABLE are already removed from the list, so this avoids duplicates.
n_before = len(all_elements)
all_elements = reconstructed_elements
print(f"📦 all_elements: {n_before} → {len(all_elements)} elements (absorbed {n_before - len(all_elements)} sub-elements into RECONSTRUCTED_TABLEs)")

### 2.2.1 Save the unmasked JSON file with reconstruction.

In [22]:
docling_reconstructed_json_path = DOCLING_OUTPUT_DIR / f"{PDF_PATH.stem}_full_reconstructed_layout.json"

with open(docling_reconstructed_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(PDF_PATH), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc.pages.items()},
        "elements": all_elements
    }, f, indent=2)

types = Counter([el['type'] for el in all_elements])
print(f"\n✅ Original PDF: {len(all_elements)} elements")
print(f"   Tables: {types.get('TABLE', 0)}")
print(f"   Figures: {types.get('PICTURE', 0)}")
print(f"   Captions: {types.get('CAPTION', 0)}")
print(f"💾 {docling_reconstructed_json_path}")


✅ Original PDF: 407 elements
   Tables: 2
   Figures: 9
   Captions: 15
💾 out/docling_full/PMC1448691_his_2369_full_reconstructed_layout.json


### 2.3 Visualize Docling Elements and Reconstructed Tables 

In [26]:
docling_json_path = DOCLING_OUTPUT_DIR / f"{PDF_PATH.stem}_full_layout.json"
docling_reconstructed_json_path = DOCLING_OUTPUT_DIR / f"{PDF_PATH.stem}_full_reconstructed_layout.json"

original_output_path = f"{PDF_PATH.stem}_full_layout.pdf"
reconstructed_output_path = f"{PDF_PATH.stem}_full_reconstructed_layout.pdf"

visualize_pdfs(str(PDF_PATH), str(docling_json_path), output_path=original_output_path)
visualize_pdfs(str(PDF_PATH), str(docling_reconstructed_json_path), output_path=reconstructed_output_path)

Loaded: 422 elements

PDF: PMC1448691_his_2369.pdf
Pages: 24

out/visualization/PMC1448691_his_2369_full_layout.pdf

Total elements drawn: 422

Element type breakdown:
  TEXT                :  217
  LIST_ITEM           :  157
  SECTION_HEADER      :   20
  CAPTION             :   15
  PICTURE             :    9
  FOOTNOTE            :    2
  TABLE               :    2

✅ Visualization saved to: out/visualization/PMC1448691_his_2369_full_layout.pdf

Loaded: 407 elements

PDF: PMC1448691_his_2369.pdf
Pages: 24

out/visualization/PMC1448691_his_2369_full_reconstructed_layout.pdf

Total elements drawn: 407

Element type breakdown:
  TEXT                :  201
  LIST_ITEM           :  153
  SECTION_HEADER      :   20
  CAPTION             :   15
  PICTURE             :    9
  RECONSTRUCTED_TABLE :    5
  FOOTNOTE            :    2
  TABLE               :    2

✅ Visualization saved to: out/visualization/PMC1448691_his_2369_full_reconstructed_layout.pdf



## 3. PDF Masking

### 3.1 Create Masked PDF

Replace tables and figures with white rectangles to create a text-only PDF.

In [27]:
print("🔄 Masking tables and figures...")
masked_pdf_path, _, masked_elements = process_pdf_with_masking(
    pdf_path=PDF_PATH,
    json_path=docling_json_path,
    output_dir=MASKED_PDF_DIR
)
print(f"\n✅ Masked PDF: {masked_pdf_path}")
print(f"🚫 Masked: {len(masked_elements)} elements (tables/figures/captions)")

INFO: Processing: PMC1448691_his_2369.pdf
INFO:   Reconstructing tables from elements...
INFO:   Found 31 maskable elements (including 5 reconstructed tables)
INFO:     ✓ Masked element CAPTION with text = Table 1. B-cell cutaneous lymphoma. Learning from the Workshop on page 2
INFO:     ✓ Masked element RECONSTRUCTED_TABLE with text = NO_TEXT on page 2
INFO:     ✓ Masked element PICTURE with text = None on page 3
INFO:     ✓ Masked element CAPTION with text = Figure 1. a-f, Primary cutaneous follicle centre lymphoma. a, Nodular pattern. b, Centroblastic predominance. c, CD20. d, CD10. e, Bcl-2. f, Ki67. g-j, Primary cutaneous diffuse large B-cell lymphoma, leg type. g,h, morphology, H&E. i, CD20. j, Ki67. Cases contributed by C. Girardet (A6) and R. S. Robertorye (A8). on page 3
INFO:     ✓ Masked element TABLE with text = None on page 5
INFO:     ✓ Masked element CAPTION with text = Table 2. Comparison of the immunophenotype of normal and tumoral plasmacytoid dendritic cells (PDC) on

🔄 Masking tables and figures...


INFO:   ✓ Saved masked PDF: out/masked_pdfs/PMC1448691_his_2369_masked.pdf
INFO:   ✓ Masked 31 regions
INFO:   ✓ Extracted 374 text elements




✅ Masked PDF: out/masked_pdfs/PMC1448691_his_2369_masked.pdf
🚫 Masked: 31 elements (tables/figures/captions)


### 3.2 Extract Layout from Masked PDF

Run Docling on the masked PDF to extract clean text elements without tables/figures.

In [29]:
print(f"🔄 Extracting layout from MASKED PDF...")
result_masked = converter.convert(str(masked_pdf_path))
doc_masked = result_masked.document

masked_pdf_elements = []
for element, level in doc_masked.iterate_items():
    label = str(getattr(element, "label", "UNKNOWN")).split('.')[-1].upper()
    if not (hasattr(element, 'prov') and element.prov):
        continue
    prov = element.prov[0]
    bbox = prov.bbox
    text = ""
    if hasattr(element, 'text'):
        text = element.text
    elif hasattr(element, 'caption') and element.caption:
        text = element.caption.text
    masked_pdf_elements.append({
        "type": label,
        "page": prov.page_no,
        "level": level,
        "bbox": {"x1": bbox.l, "y1": bbox.t, "x2": bbox.r, "y2": bbox.b},
        "text": text.strip() if text else None
    })

masked_json_path = DOCLING_OUTPUT_DIR / f"{masked_pdf_path.stem}_full_reconstructed_layout.json"
with open(masked_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(masked_pdf_path), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc_masked.pages.items()},
        "elements": masked_pdf_elements
    }, f, indent=2)

masked_types = Counter([el['type'] for el in masked_pdf_elements])
print(f"\n✅ Masked PDF: {len(masked_pdf_elements)} elements")
print(f"\n📊 COMPARISON:")
print(f"   Original PDF: {len(all_elements)} elements")
print(f"   Masked PDF:   {len(masked_pdf_elements)} elements")
print(f"   Removed:      {len(all_elements) - len(masked_pdf_elements)} elements")
print(f"\n🔍 Masked PDF element types:")
for t, c in masked_types.most_common():
    print(f"   {t}: {c}")
print(f"\n💾 {masked_json_path}")

# Extract text elements from masked PDF for stitching
text_element_types = {'TEXT', 'PARAGRAPH', 'SECTION_HEADER', 'TITLE', 'LIST', 'LIST_ITEM'}
text_elements = [el for el in masked_pdf_elements if el.get('type') in text_element_types]
print(f"\n📝 Text elements for processing: {len(text_elements)}")

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Processing document PMC1448691_his_2369_masked.pdf


🔄 Extracting layout from MASKED PDF...


INFO: Finished converting document PMC1448691_his_2369_masked.pdf in 15.24 sec.



✅ Masked PDF: 280 elements

📊 COMPARISON:
   Original PDF: 407 elements
   Masked PDF:   280 elements
   Removed:      127 elements

🔍 Masked PDF element types:
   LIST_ITEM: 153
   TEXT: 107
   SECTION_HEADER: 19
   FOOTNOTE: 1

💾 out/docling_full/PMC1448691_his_2369_masked_full_reconstructed_layout.json

📝 Text elements for processing: 279


## 4. Text Processing

### 4.1 Shared Extraction Utilities

A single `extract_text()` function handles all combinations of reconstruction and masking.

In [89]:
from collections import defaultdict

SKIP_TYPES = {'TABLE', 'FIGURE', 'PICTURE', 'CAPTION', 'RECONSTRUCTED_TABLE', 'FOOTNOTE'}

def build_table_bboxes(elements, types=('TABLE', 'RECONSTRUCTED_TABLE')):
    """Per-page index of bboxes for the given element types."""
    index = defaultdict(list)
    for el in elements:
        if el.get('type') in types:
            page, bbox = el.get('page'), el.get('bbox')
            if page and bbox:
                index[page].append(bbox)
    return index

def build_picture_pages(elements):
    """Set of page numbers that contain at least one PICTURE element."""
    return {el['page'] for el in elements if el.get('type') == 'PICTURE' and el.get('page')}

def bbox_overlaps(a, b):
    """True if bbox a overlaps bbox b."""
    return not (a['x1'] >= b['x2'] or a['x2'] <= b['x1'] or
                a['y2'] >= b['y1'] or a['y1'] <= b['y2'])

def centroid_inside(a, b):
    """True if the centroid of bbox a falls inside bbox b."""
    cx = (a['x1'] + a['x2']) / 2
    cy = (a['y1'] + a['y2']) / 2
    return (b['x1'] < cx < b['x2'] and
            min(b['y1'], b['y2']) < cy < max(b['y1'], b['y2']))

def extract_text(elements, table_bboxes=None, use_centroid=False):
    """Extract and stitch hierarchical text from a list of Docling elements.

    Args:
        elements:      List of element dicts (type, page, level, bbox, text).
        table_bboxes:  Optional {page: [bbox, ...]} – elements are skipped if
                       they overlap any of these bboxes.
        use_centroid:  If True, only skip an element when its centroid falls
                       inside a table/figure bbox (less aggressive than full overlap).

    Returns:
        (stitched_by_path, n_skipped)
    """
    overlap_fn = centroid_inside if use_centroid else bbox_overlaps
    picture_pages = build_picture_pages(elements)

    hierarchy = {}
    by_path   = defaultdict(list)
    skipped   = 0

    for el in elements:
        etype = el.get('type', '')
        level = el.get('level', 0)
        text  = (el.get('text') or '').strip()
        if not text:
            continue

        if etype == 'SECTION_HEADER':
            hierarchy[level] = text
            hierarchy = {k: v for k, v in hierarchy.items() if k <= level}
        elif etype not in SKIP_TYPES:
            # Drop single-character tokens on pages with figures — these are
            # panel labels (a, b, c …) that Docling OCRs just outside the PICTURE bbox.
            if len(text) == 1 and el.get('page') in picture_pages:
                skipped += 1
                continue
            if table_bboxes:
                page = el.get('page')
                bbox = el.get('bbox')
                if page and bbox and any(overlap_fn(bbox, tb) for tb in table_bboxes.get(page, [])):
                    skipped += 1
                    continue
            path_parts = [hierarchy[k] for k in sorted(hierarchy) if hierarchy.get(k)]
            by_path[' > '.join(path_parts) or 'Root'].append(text)

    stitcher = ContextAwareStitcher()
    stitched = {
        path: stitcher.reconstruct_paragraphs([remove_citations(t) for t in texts])
        for path, texts in by_path.items()
    }
    return stitched, skipped

print("✅ extract_text() ready")

✅ extract_text() ready


### 4.2 Run All Combinations

| Combination | Elements | Table filtering |
|---|---|---|
| `direct \| raw` | original Docling output | SKIP_TYPES + TABLE bbox |
| `direct \| reconstructed` | lists merged into RECONSTRUCTED_TABLE | SKIP_TYPES + TABLE + RECONSTRUCTED_TABLE bbox |
| `masked` | re-extracted from white-masked PDF | none needed (regions are blank) |

In [90]:
COMBINATIONS = {
    'direct | raw': (
        all_elements_raw,
        build_table_bboxes(all_elements_raw, types=('TABLE', 'PICTURE'))
    ),
    'direct | reconstructed': (
        all_elements,
        build_table_bboxes(all_elements, types=('TABLE', 'RECONSTRUCTED_TABLE', 'PICTURE'))
    ),
    'masked': (text_elements, None),   # masked PDF already has blank regions
}

results = {}  # name -> stitched_by_path
print("Running text extraction combinations:")
print(f"  {'Combination':<26s} {'Paths':>6s} {'Paras':>6s} {'Skipped':>8s}")
print(f"  {'-'*50}")
for name, (elements, bboxes) in COMBINATIONS.items():
    stitched, skipped = extract_text(elements, bboxes)
    results[name] = stitched
    n_paths = len(stitched)
    n_paras = sum(len(v) for v in stitched.values())
    print(f"  {name:<26s} {n_paths:>6d} {n_paras:>6d} {skipped:>8d}")

Running text extraction combinations:
  Combination                 Paths  Paras  Skipped
  --------------------------------------------------
  direct | raw                   18    233       96
  direct | reconstructed         18    229       93
  masked                         17    228        0


### 4.3 Compare Results

In [91]:
import difflib

BASELINE = 'masked'   # change to any key in results to shift the reference

names = list(results.keys())
others = [n for n in names if n != BASELINE]
all_paths = sorted(set().union(*[set(results[n]) for n in names]))

# ── Summary table ──────────────────────────────────────────────────────────────
header = f"{'Path':<55s}" + "".join(f" {n[:10]:>10s}" for n in names)
print(header)
print("-" * len(header))
for path in all_paths:
    counts = [len(results[n].get(path, [])) for n in names]
    marker = " *" if len(set(counts)) > 1 else ""
    print(f"{path[:53]:<55s}" + "".join(f" {c:>10d}" for c in counts) + marker)

# ── Text diffs vs baseline ─────────────────────────────────────────────────────
for other in others:
    print(f"\n{'='*80}")
    print(f"DIFF  {BASELINE!r}  vs  {other!r}")
    print('='*80)
    any_diff = False
    for path in all_paths:
        a = results[BASELINE].get(path, [])
        b = results[other].get(path, [])
        if a == b:
            continue
        any_diff = True
        print(f"\n  [{path}]")
        for line in difflib.unified_diff(a, b, lineterm='', fromfile=BASELINE, tofile=other):
            if line.startswith('+') and not line.startswith('+++'):
                print(f"    + {line[1:][:120]}")
            elif line.startswith('-') and not line.startswith('---'):
                print(f"    - {line[1:][:120]}")
    if not any_diff:
        print("  (identical)")

Path                                                    direct | r direct | r     masked
----------------------------------------------------------------------------------------
Anaplastic large-cell lymphoma and its differential d           18         17         18 *
Highlights of cutaneous lymphomas                               10         10         10
Introduction                                                     3          3          3
Large B-cell lymphomas of terminally differentiated B            1          1          1
Lymphomas at other extranodal sites                              6          6          5 *
Mucosa-associated lymphomas                                      1          1          1
Other lymphoproliferative disorders in immunocompromi            6          5          9 *
Plasmacytoid dendritic cell tumours                             11         11         11
Pyothorax-associated lymphoma                                    4          4          0 *
References   

### 4.4 Save Results and Diffs

In [92]:
# Save each combination's text
for name, stitched in results.items():
    slug = name.replace(' ', '_').replace('|', '-').replace('/', '-')
    out_path = TEXT_OUTPUT_DIR / f"{PMCID}_{slug}.txt"
    with open(out_path, 'w') as f:
        f.write(f"Document: {PMCID}  |  combination: {name}\n{'='*80}\n\n")
        for path, paragraphs in sorted(stitched.items()):
            f.write(f"[{path}]\n{'-'*80}\n")
            for p in paragraphs:
                if p.strip():
                    f.write(f"{p}\n\n")
            f.write("\n")
    print(f"💾 {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

# ── Centroid vs bbox comparison ────────────────────────────────────────────────
# Run direct|reconstructed with both filtering modes and save to separate files
_bboxes = build_table_bboxes(all_elements, types=('TABLE', 'RECONSTRUCTED_TABLE', 'PICTURE'))

for label, use_centroid in [('bbox', False), ('centroid', True)]:
    stitched, skipped = extract_text(all_elements, _bboxes, use_centroid=use_centroid)
    out_path = TEXT_OUTPUT_DIR / f"{PMCID}_direct_reconstructed_{label}.txt"
    with open(out_path, 'w') as f:
        f.write(f"Document: {PMCID}  |  direct|reconstructed  |  filter: {label}  |  skipped: {skipped}\n{'='*80}\n\n")
        for path, paragraphs in sorted(stitched.items()):
            f.write(f"[{path}]\n{'-'*80}\n")
            for p in paragraphs:
                if p.strip():
                    f.write(f"{p}\n\n")
            f.write("\n")
    print(f"💾 {out_path} ({out_path.stat().st_size / 1024:.1f} KB)  (skipped {skipped} elements)")

# Save diffs vs baseline
diff_path = TEXT_OUTPUT_DIR / f"{PMCID}_diffs_{BASELINE.replace(' ','_')}_vs_others.txt"
with open(diff_path, 'w') as f:
    f.write(f"DIFFS — baseline: {BASELINE!r}  |  {PMCID}\n{'='*80}\n\n")
    for other in others:
        f.write(f"\n{'='*80}\n")
        f.write(f"DIFF  {BASELINE!r}  vs  {other!r}\n")
        f.write(f"{'='*80}\n")
        any_diff = False
        for path in all_paths:
            a = results[BASELINE].get(path, [])
            b = results[other].get(path, [])
            if a == b:
                continue
            any_diff = True
            f.write(f"\n  [{path}]\n")
            for line in difflib.unified_diff(a, b, lineterm='', fromfile=BASELINE, tofile=other):
                if line.startswith('+') and not line.startswith('+++'):
                    f.write(f"    + {line[1:]}\n")
                elif line.startswith('-') and not line.startswith('---'):
                    f.write(f"    - {line[1:]}\n")
        if not any_diff:
            f.write("  (identical)\n")
print(f"💾 {diff_path} ({diff_path.stat().st_size / 1024:.1f} KB)")

💾 out/text/PMC1448691_direct_-_raw.txt (90.6 KB)
💾 out/text/PMC1448691_direct_-_reconstructed.txt (89.7 KB)
💾 out/text/PMC1448691_masked.txt (89.9 KB)
💾 out/text/PMC1448691_direct_reconstructed_bbox.txt (89.7 KB)  (skipped 93 elements)
💾 out/text/PMC1448691_direct_reconstructed_centroid.txt (89.7 KB)  (skipped 93 elements)
💾 out/text/PMC1448691_diffs_masked_vs_others.txt (32.7 KB)


## 5. Media Extraction

### 5.1 Crop Table and Figure Images

Extract table and figure regions from the original PDF as high-resolution images.

In [ ]:
import re
import numpy as np

FIG_NUM_RE = re.compile(r'^figure\s+(\d+)', re.IGNORECASE)
TAB_NUM_RE = re.compile(r'^table\s+(\d+)', re.IGNORECASE)

def find_nearest_caption(el, caption_elements):
    """Find the spatially closest CAPTION on the same page."""
    page = el.get('page')
    bbox = el.get('bbox')
    if not page or not bbox:
        return None
    same_page = [c for c in caption_elements if c.get('page') == page]
    if not same_page:
        return None
    cx = (bbox['x1'] + bbox['x2']) / 2
    cy = (bbox['y1'] + bbox['y2']) / 2
    return min(same_page, key=lambda c: (
        ((c['bbox']['x1'] + c['bbox']['x2']) / 2 - cx) ** 2 +
        ((c['bbox']['y1'] + c['bbox']['y2']) / 2 - cy) ** 2
    ))

def parse_number(text, pattern):
    m = pattern.match(text or '')
    return int(m.group(1)) if m else None

def union_bbox(a, b):
    """Return the bounding box that covers both a and b."""
    return {
        'x1': min(a['x1'], b['x1']),
        'x2': max(a['x2'], b['x2']),
        # In Docling PDF coords higher y = higher on page; use max/min to expand in both directions
        'y1': max(max(a['y1'], a['y2']), max(b['y1'], b['y2'])),
        'y2': min(min(a['y1'], a['y2']), min(b['y1'], b['y2'])),
    }

def detect_panel_splits(page, bbox, page_height, min_gap_px=8, white_threshold=248):
    """
    Detect side-by-side panels within a figure by finding white vertical gaps.
    Renders the figure region at 1x to find column gaps, then maps them back
    to PDF coordinates.
    Returns a list of sub-bboxes (just [bbox] if no split is found).
    """
    b = bbox
    rect = fitz.Rect(b['x1'], page_height - max(b['y1'], b['y2']),
                     b['x2'], page_height - min(b['y1'], b['y2']))
    pix = page.get_pixmap(clip=rect, matrix=fitz.Matrix(1, 1))
    if pix.width == 0 or pix.height == 0:
        return [bbox]

    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    col_means = (img[..., :3].mean(axis=(0, 2)) if pix.n >= 3
                 else img[..., 0].mean(axis=0).astype(float))

    is_white = col_means > white_threshold

    gap_centers = []
    in_gap, gap_start = False, 0
    for i, w in enumerate(is_white):
        if w and not in_gap:
            in_gap, gap_start = True, i
        elif not w and in_gap:
            in_gap = False
            if i - gap_start >= min_gap_px:
                gap_centers.append((gap_start + i) / 2 / pix.width)

    if not gap_centers:
        return [bbox]

    w_total = b['x2'] - b['x1']
    xs = [b['x1']] + [b['x1'] + g * w_total for g in gap_centers] + [b['x2']]
    return [{**b, 'x1': xs[i], 'x2': xs[i + 1]} for i in range(len(xs) - 1)]


all_captions = [el for el in masked_elements if el.get('type') == 'CAPTION']

table_data_raw, figure_data = [], []

for el in masked_elements:
    t = el.get('type')

    if t in ['TABLE', 'RECONSTRUCTED_TABLE']:
        raw_caption = el.get('caption') or ''
        if not raw_caption:
            nearest = find_nearest_caption(el, all_captions)
            raw_caption = nearest.get('text', '') if nearest else ''
        num = parse_number(raw_caption, TAB_NUM_RE) or (len(table_data_raw) + 1)
        table_data_raw.append({'table_id': str(num), 'caption': raw_caption or f'Table {num}',
                               'page': el.get('page'), 'bbox': el.get('bbox'), 'type': t.lower()})

    elif t in ['FIGURE', 'PICTURE']:
        nearest = find_nearest_caption(el, all_captions)
        raw_caption = nearest.get('text', '') if nearest else ''
        num = parse_number(raw_caption, FIG_NUM_RE) or (len(figure_data) + 1)
        figure_data.append({'figure_id': str(num), 'caption': raw_caption or f'Figure {num}',
                            'page': el.get('page'), 'bbox': el.get('bbox'), 'type': t.lower()})

# Deduplicate tables that share the same ID (e.g. a Docling TABLE + RECONSTRUCTED_TABLE
# covering different parts of the same table).  Merge their bboxes so the crop includes
# the full table, and prefer the richer caption.
merged_tables = {}
for t in table_data_raw:
    tid = t['table_id']
    if tid not in merged_tables:
        merged_tables[tid] = t.copy()
    else:
        existing = merged_tables[tid]
        if existing['bbox'] and t['bbox']:
            existing['bbox'] = union_bbox(existing['bbox'], t['bbox'])
        # Keep the longer caption (the RECONSTRUCTED_TABLE one is usually richer)
        if len(t['caption'] or '') > len(existing['caption'] or ''):
            existing['caption'] = t['caption']
        if existing['type'] == 'table' and t['type'] == 'reconstructed_table':
            existing['type'] = 'reconstructed_table'
        print(f"   ⚠️  Merged duplicate Table {tid} ({existing['type']} + {t['type']})")
table_data = list(merged_tables.values())

doc = fitz.open(str(PDF_PATH))

for t in table_data:
    if t['page'] and t['bbox']:
        p = doc[t['page'] - 1]
        h = p.rect.height
        b = t['bbox']
        r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
        pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
        path = TABLES_DIR / f"{PMCID}_table_{t['table_id']}.png"
        pix.save(str(path))
        t['image_path'] = str(path)

for f in figure_data:
    if not (f['page'] and f['bbox']):
        continue
    p = doc[f['page'] - 1]
    h = p.rect.height

    sub_bboxes = detect_panel_splits(p, f['bbox'], h)

    if len(sub_bboxes) == 1:
        b = f['bbox']
        r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
        pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
        path = FIGURES_DIR / f"{PMCID}_figure_{f['figure_id']}.png"
        pix.save(str(path))
        f['image_path'] = str(path)
        f['panels'] = 1
    else:
        paths = []
        for j, b in enumerate(sub_bboxes):
            suffix = chr(ord('a') + j)
            r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
            pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
            path = FIGURES_DIR / f"{PMCID}_figure_{f['figure_id']}{suffix}.png"
            pix.save(str(path))
            paths.append(str(path))
        f['image_path'] = paths
        f['panels'] = len(sub_bboxes)

doc.close()
print(f"🖼️  Cropped {len(table_data)} tables, {len(figure_data)} figures")
for td in table_data:
    print(f"   Table {td['table_id']}: {(td['caption'] or '')[:80]}")
for fd in figure_data:
    panels = fd.get('panels', 1)
    panel_str = f" ({panels} panels)" if panels > 1 else ""
    print(f"   Figure {fd['figure_id']}{panel_str}: {(fd['caption'] or '')[:80]}")

### 5.2 Save Metadata

Export table and figure metadata as JSON files.

In [ ]:
if table_data:
    with open(TABLES_DIR / f"{PMCID}_tables.json", 'w') as f:
        json.dump(table_data, f, indent=2)
if figure_data:
    with open(FIGURES_DIR / f"{PMCID}_figures.json", 'w') as f:
        json.dump(figure_data, f, indent=2)
print("💾 Metadata saved")

## 7. Database Ingestion

### 7.1 Prepare Data for Database

Convert stitched text by path into database-compatible hierarchical structure.

In [ ]:
# Import database modules
from database import get_db_connection, Document, TextElement, Figure, Table
from database.models import TextElementFigureReference, TextElementTableReference

# Prepare hierarchical elements for database
# Need to convert stitched_by_path back into individual text elements with hierarchical info
db_text_elements = []

for path_string, stitched_paras in stitched_by_path.items():
    # Build path_list from path_string
    if path_string == 'Root':
        path_list = []
        depth = 0
    else:
        path_list = [part.strip() for part in path_string.split(' > ')]
        depth = len(path_list)
    
    # Each stitched paragraph becomes one text element
    for para in stitched_paras:
        if para.strip():
            # Note: We don't have page info or references at this stage
            # In production, you'd track these during the initial grouping
            db_text_elements.append({
                'path_list': path_list,
                'path_string': path_string,
                'depth': depth,
                'text': para,
                'references': {}  # Would be populated if we tracked them
            })

print(f"📦 Prepared {len(db_text_elements)} text elements for database")
print(f"   Organized into {len(stitched_by_path)} hierarchical paths")

# Preview first few elements
print(f"\n🔍 Sample elements:")
for i, elem in enumerate(db_text_elements[:3]):
    print(f"   {i+1}. Path: {elem['path_string']}")
    print(f"      Text: {elem['text'][:80]}...")
    print()

### 7.2 Ingest to PostgreSQL Database

Save document with hierarchical text elements, figures, and tables to database.

In [ ]:
# Database ingestion (similar to comprehensive_ingest.py)
db = get_db_connection()

try:
    with db.session_scope() as session:
        # Check if document already exists
        existing = session.query(Document).filter_by(pmcid=PMCID).first()
        
        if existing:
            print(f"⚠️  Document {PMCID} already exists in database")
            print(f"   Use force=True to re-ingest or delete manually")
            force_reingest = False  # Set to True to delete and re-create
            
            if force_reingest:
                print(f"🗑️  Deleting existing document...")
                session.delete(existing)
                session.flush()
            else:
                print(f"   Skipping ingestion...")
                raise Exception("Document exists - set force_reingest=True to overwrite")
        
        # Create document record
        doc = Document(
            pmcid=PMCID,
            filename=PDF_PATH.name,
            file_path=str(PDF_PATH.absolute()),
            title=f"Document {PMCID}",  # Could extract from PDF if available
            journal=None,
            publication_year=None,
            text_source='pdf'
        )
        session.add(doc)
        session.flush()
        print(f"✅ Created document: {PMCID}")
        
        # Add text elements with position tracking
        path_counts = defaultdict(int)
        for elem in db_text_elements:
            path_string = elem['path_string']
            position = path_counts[path_string]
            path_counts[path_string] += 1
            
            # Create unique path: {pmcid}/{path_string}/{position}
            unique_path = f"{PMCID}/{path_string}/{position}" if path_string else f"{PMCID}/(Root)/{position}"
            
            text_elem = TextElement(
                unique_path=unique_path,
                document_id=doc.id,
                path_list=elem['path_list'],
                path_string=path_string,
                depth=elem['depth'],
                text_content=elem['text'],
                position_in_section=position,
                references=elem.get('references', {})
            )
            session.add(text_elem)
        
        session.flush()
        print(f"✅ Added {len(db_text_elements)} text elements")
        
        # Add figures
        for fig in figure_data:
            image_filename = None
            image_path = fig.get('image_path')
            if image_path:
                image_filename = Path(image_path).name
            
            figure = Figure(
                document_id=doc.id,
                figure_id=fig['figure_id'],
                figure_label=f"Figure {fig['figure_id']}",
                figure_number=fig['figure_id'],
                caption_text=fig.get('caption'),
                image_filename=image_filename,
                image_path=image_path
            )
            session.add(figure)
        
        session.flush()
        print(f"✅ Added {len(figure_data)} figures")
        
        # Add tables
        for tbl in table_data:
            image_filename = None
            image_path = tbl.get('image_path')
            if image_path:
                image_filename = Path(image_path).name
            
            table = Table(
                document_id=doc.id,
                table_id=tbl['table_id'],
                table_label=f"Table {tbl['table_id']}",
                table_number=tbl['table_id'],
                caption_text=tbl.get('caption'),
                image_filename=image_filename,
                image_path=image_path
            )
            session.add(table)
        
        session.flush()
        print(f"✅ Added {len(table_data)} tables")
        
        # Note: Figure/table references would be created here if we tracked them
        # See comprehensive_ingest.py lines 926-968 for reference creation logic
        
        print(f"\n🎉 Successfully ingested {PMCID} to database!")
        print(f"   Document ID: {doc.id}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

## 8. Summary

### 8.1 Pipeline Statistics and Output Files

In [ ]:
print("="*80)
print("📊 PIPELINE COMPLETE")
print("="*80)
print(f"\n📄 Input: {PDF_PATH.name}")
print(f"   PMCID: {PMCID}")
print(f"\n📊 Processing Statistics:")
print(f"   Original PDF elements:  {len(all_elements)}")
print(f"   Masked PDF elements:    {len(masked_pdf_elements)}")
print(f"   Elements removed:       {len(all_elements) - len(masked_pdf_elements)}")
print(f"   Text elements extracted: {len(text_elements)}")
print(f"   Hierarchical paths:     {len(stitched_by_path)}")
print(f"   Stitched paragraphs:    {total_stitched}")
print(f"   DB text elements:       {len(db_text_elements)}")
print(f"\n📝 Output Files:")
print(f"   Original JSON:     {docling_json_path.name}")
print(f"   Masked PDF:        {masked_pdf_path.name}")
print(f"   Masked PDF JSON:   {masked_json_path.name}")
print(f"   Text file:         {text_path.name} ({text_path.stat().st_size / 1024:.1f} KB)")
print(f"   Tables metadata:   {TABLES_DIR / f'{PMCID}_tables.json'}")
print(f"   Figures metadata:  {FIGURES_DIR / f'{PMCID}_figures.json'}")
print(f"\n🖼️  Media Extraction:")
print(f"   Tables:    {len(table_data)} images")
print(f"   Figures:   {len(figure_data)} images")
print(f"\n💾 Database Ingestion:")
print(f"   Documents:      1")
print(f"   Text elements: {len(db_text_elements)}")
print(f"   Figures:       {len(figure_data)}")
print(f"   Tables:        {len(table_data)}")
print("\n✅ Done!")
print("\n💡 Key Features:")
print("   ✓ Hierarchical text organization by section paths")
print("   ✓ Paragraph stitching to join split text")
print("   ✓ Citation removal from text content")
print("   ✓ Table reconstruction from captions")
print("   ✓ Clean text extraction via PDF masking")
print("   ✓ PostgreSQL database storage with full relationships")